# Programa 2 · Planificador de producción

El programa representa una producción mediante una clase y evalúa si puede realizarse con el presupuesto y el tiempo disponibles.

**Estructuras que aparecen:** tipos, listas, bucles, funciones, condicionales, una clase, un objeto y métodos.

## 1. Clase `Produccion`

El objeto guardará los datos generales y la lista de materiales. Sus métodos realizarán los cálculos de materiales, mano de obra, tiempo y viabilidad.

In [5]:
class Produccion:
    def __init__(
        self,
        nombre: str,
        unidades: int,
        presupuesto: float,
        horas_disponibles: float,
    ) -> None:
        self.nombre: str = nombre
        self.unidades: int = unidades
        self.presupuesto: float = presupuesto
        self.horas_disponibles: float = horas_disponibles
        self.materiales: list[tuple[str, float, float]] = []

    def agregar_material(
        self,
        nombre: str,
        cantidad_por_unidad: float,
        precio: float,
    ) -> None:
        self.materiales.append((nombre, cantidad_por_unidad, precio))

    def calcular_materiales(self) -> float:
        total: float = 0.0

        for _, cantidad, precio in self.materiales:
            total += cantidad * precio * self.unidades

        return total

    def calcular_mano_obra(
        self,
        horas_por_unidad: float,
        valor_hora: float,
    ) -> float:
        return horas_por_unidad * valor_hora * self.unidades

    def calcular_total(
        self,
        horas_por_unidad: float,
        valor_hora: float,
    ) -> float:
        costo_materiales: float = self.calcular_materiales()
        costo_mano_obra: float = self.calcular_mano_obra(
            horas_por_unidad,
            valor_hora,
        )
        return costo_materiales + costo_mano_obra

    def calcular_horas_necesarias(self, horas_por_unidad: float) -> float:
        return horas_por_unidad * self.unidades


    def calcular_ganancia(self, precio_venta: float,
                     horas_por_unidad: float,
                     valor_hora: float) -> float:

        ingresos = precio_venta * self.unidades
        costos = self.calcular_total(horas_por_unidad, valor_hora)

        return ingresos - costos


    def calcular_margen(self, precio_venta: float,
                   horas_por_unidad: float,
                   valor_hora: float) -> float:

        ingresos = precio_venta * self.unidades

        if ingresos == 0:
            return 0.0

        ganancia = self.calcular_ganancia(
        precio_venta, horas_por_unidad, valor_hora
        )

        return ganancia / ingresos * 100

    def calcular_maximo_unidades(self, horas_por_unidad: float) -> int:
        if horas_por_unidad <= 0:
            raise ValueError("Las horas por unidad deben ser mayores que cero")

        return int(self.horas_disponibles // horas_por_unidad)

    
    def evaluar(
        self,
        horas_por_unidad: float,
        valor_hora: float,
    ) -> str:
        costo: float = self.calcular_total(horas_por_unidad, valor_hora)
        horas_necesarias: float = self.calcular_horas_necesarias(
            horas_por_unidad
        )

        presupuesto_ok: bool = costo <= self.presupuesto
        tiempo_ok: bool = horas_necesarias <= self.horas_disponibles

        if presupuesto_ok and tiempo_ok:
            return "Producción viable por presupuesto y tiempo"
        if not presupuesto_ok and not tiempo_ok:
            return "Producción no viable: revisar presupuesto y tiempo"
        if not presupuesto_ok:
            return "Producción no viable: revisar presupuesto"
        return "Producción no viable: revisar tiempo disponible"

## 2. Creación del objeto

Creamos una producción concreta y luego usamos sus métodos para agregar los materiales.

In [6]:
produccion: Produccion = Produccion(
    nombre="Lámpara modular",
    unidades=20,
    presupuesto=1_250_000.0,
    horas_disponibles=55.0,
)

produccion.agregar_material("Madera", 0.8, 24_000.0)
produccion.agregar_material("Cable", 1.5, 2_800.0)
produccion.agregar_material("Portalámpara", 1.0, 5_600.0)

horas_por_unidad: float = 2.2
valor_hora: float = 7_500.0

## 3. Cálculos e informe

In [7]:
costo_materiales: float = produccion.calcular_materiales()
costo_mano_obra: float = produccion.calcular_mano_obra(
    horas_por_unidad,
    valor_hora,
)
costo_total: float = produccion.calcular_total(
    horas_por_unidad,
    valor_hora,
)
horas_necesarias: float = produccion.calcular_horas_necesarias(
    horas_por_unidad
)
estado: str = produccion.evaluar(
    horas_por_unidad,
    valor_hora,
)

print(f"PRODUCCIÓN: {produccion.nombre}")
print(f"Unidades: {produccion.unidades}")
print("-" * 52)

for nombre, cantidad, precio in produccion.materiales:
    parcial: float = cantidad * precio * produccion.unidades
    print(f"{nombre:<18} ${parcial:>14,.2f}")

print("-" * 52)
print(f"Materiales:          ${costo_materiales:>14,.2f}")
print(f"Mano de obra:        ${costo_mano_obra:>14,.2f}")
print(f"Costo total:         ${costo_total:>14,.2f}")
print(f"Presupuesto:         ${produccion.presupuesto:>14,.2f}")
print(f"Horas necesarias:    {horas_necesarias:>14.1f}")
print(f"Horas disponibles:   {produccion.horas_disponibles:>14.1f}")
print(f"Estado: {estado}")


precio_venta: float = 60000.0

ganancia = produccion.calcular_ganancia(
    precio_venta, horas_por_unidad, valor_hora
)

margen = produccion.calcular_margen(
    precio_venta, horas_por_unidad, valor_hora
)

maximo = produccion.calcular_maximo_unidades(horas_por_unidad)

print(f"Ingresos esperados: ${precio_venta * produccion.unidades:,.2f}")
print(f"Ganancia esperada: ${ganancia:,.2f}")
print(f"Margen esperado: {margen:.2f}%")
print(f"Máximo de unidades según tiempo: {maximo}")

PRODUCCIÓN: Lámpara modular
Unidades: 20
----------------------------------------------------
Madera             $    384,000.00
Cable              $     84,000.00
Portalámpara       $    112,000.00
----------------------------------------------------
Materiales:          $    580,000.00
Mano de obra:        $    330,000.00
Costo total:         $    910,000.00
Presupuesto:         $  1,250,000.00
Horas necesarias:              44.0
Horas disponibles:             55.0
Estado: Producción viable por presupuesto y tiempo
Ingresos esperados: $1,200,000.00
Ganancia esperada: $290,000.00
Margen esperado: 24.17%
Máximo de unidades según tiempo: 24


## Para expandir

1. Agregá el precio de venta y calculá el margen esperado.
2. Creá tres objetos `Produccion` con distintas cantidades de unida
3. des y compará su viabilidad.
4. Incorporá un método que calcule la cantidad máxima de unidades según el tiempo disponible.

In [8]:
cantidades = [20, 24, 30]

for unidades in cantidades:
    prueba = Produccion(
        nombre="Lámpara modular",
        unidades=unidades,
        presupuesto=1250000.0,
        horas_disponibles=55.0
    )

    prueba.agregar_material("Madera", 0.8, 24000.0)
    prueba.agregar_material("Cable", 1.5, 2800.0)
    prueba.agregar_material("Portalámpara", 1.0, 5600.0)

    print("-" * 50)
    print(f"Unidades: {prueba.unidades}")
    print(f"Costo total: ${prueba.calcular_total(horas_por_unidad, valor_hora):,.2f}")
    print(f"Horas necesarias: {prueba.calcular_horas_necesarias(horas_por_unidad):.1f}")
    print(f"Estado: {prueba.evaluar(horas_por_unidad, valor_hora)}")

--------------------------------------------------
Unidades: 20
Costo total: $910,000.00
Horas necesarias: 44.0
Estado: Producción viable por presupuesto y tiempo
--------------------------------------------------
Unidades: 24
Costo total: $1,092,000.00
Horas necesarias: 52.8
Estado: Producción viable por presupuesto y tiempo
--------------------------------------------------
Unidades: 30
Costo total: $1,365,000.00
Horas necesarias: 66.0
Estado: Producción no viable: revisar presupuesto y tiempo
